# Project 2: word embeddings (Jeff)

GloVe Twitter op de stance data. Stap 1 en 2 van het project.

## Setup

In [ ]:
import sys
from collections import Counter
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))

from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # geen HF_TOKEN-melding bij het laden van de data
import numpy as np
import part2
from src.data import load_stance_all
from src.embeddings import coverage_table, load_vocab
from src.glove_tokenizer import tokenize, tokenize_plain

GLOVE_PATH = root / "data" / "glove.twitter.27B.50d.txt"  # eerst: uv run python scripts/download_glove.py
GLOVE_DIMS = 50
WIKI_PATH = root / "data" / "glove.6B.50d.txt"  # alternatief: uv run python scripts/download_glove.py wiki

## Data

In [ ]:
stance_dataset = load_stance_all()
train_texts = stance_dataset["train"]["text"]
len(train_texts)

## GloVe Twitter laden

In [ ]:
glove = part2.clslib.PretrainedEmbeddings(str(GLOVE_PATH), GLOVE_DIMS)
len(glove.vocab)

## Stap 1: past het model bij onze data?

GloVe Twitter is getraind op tweets die eerst zijn voorbewerkt (`#feminism` → `<hashtag> feminism`, `@user` → `<user>`, ...).
We doen dezelfde voorbewerking (`src/glove_tokenizer.py`). Alternatief: GloVe 6B (Wikipedia + nieuws).

In [ ]:
example = train_texts[0]
print(example)
print(tokenize(example))

In [ ]:
twitter_vocab = load_vocab(GLOVE_PATH)
wiki_vocab = load_vocab(WIKI_PATH)
naive = lambda text: text.lower().split()

coverage_table(train_texts, {
    "Twitter, split op spaties": (naive, twitter_vocab),
    "Twitter, GloVe-voorbewerking": (tokenize, twitter_vocab),
    "Wikipedia (6B), gewone tokenisatie": (tokenize_plain, wiki_vocab),
})

Met de voorbewerking kent GloVe Twitter 98.8% van de tokens (zonder: 71%). Wikipedia mist vooral Twitter-hashtags (gamergate, lovewins, yesallwomen) die juist de stance dragen.

## Stap 2: wat legt het model vast?

Buren van stance-woorden, vergelijking met een alternatief, beperkingen.

In [ ]:
def cos_similarity(u, v):  
    """
    Return the cosine similarity (a real number) of the two words in the embedding space of our model.
    """      
    dot_product = np.dot(u, v)
    norm_A = np.linalg.norm(u)
    norm_B = np.linalg.norm(v)

    cosine_sim = dot_product / (norm_A * norm_B)
    return(cosine_sim)

def topk_ids(v, E, k=10):    
    """
    word embedding: a D-dimensional vector
    E: our model's embedding matrix as a numpy array of shape [V, D]
    k: how many nearest neighbours we want to find

    Return a list with the k word ids that are nearest to the input word embedding v
     in embedding space according to cosine similarity. 
    """   
    norms_E = np.linalg.norm(E, axis=1)  
    norms_E[norms_E == 0] = 1    
    norm_v = np.linalg.norm(v)

    cosine_sims = (E @ v) / (norms_E * norm_v)  
    return list(np.argsort(-cosine_sims)[:k])

ids = topk_ids(glove.embed_word("feminism"), glove.vectors, k=11)
[glove.word(i) for i in ids[1:]]